In [1]:
import os
os.chdir("..")

In [3]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct", trust_remote_code=True)

In [6]:
from typing import Any
from torchdata.stateful_dataloader import StatefulDataLoader

class StatefulCycleDataLoader(StatefulDataLoader):
    def __call__(self, batch_size: int) -> list[dict[str, Any]]:
        if not hasattr(self, "iterator"):
            self.iterator = iter(self)

        data_list = []
        for _ in range(batch_size):
            try:
                data = next(self.iterator)
            except StopIteration:
                self.iterator = iter(self)
                data = next(self.iterator)
            data_list.append(data)
        return data_list

## SINGLE TURN CHAT MESSAGES

In [7]:
messages = [
         {"role": "system", "content": "SYSTEM"},
         {"role": "user", "content": "QUESTION"},
         {"role": "assistant", "content": "ANSWER"},
]

In [8]:
print(tokenizer.apply_chat_template(messages, tokenize=False))

<|im_start|>system
SYSTEM<|im_end|>
<|im_start|>user
QUESTION<|im_end|>
<|im_start|>assistant
ANSWER<|im_end|>



In [11]:
tokenizer.eos_token

'<|im_end|>'

In [10]:
states = tokenizer.apply_chat_template(messages)

In [15]:
tokenizer.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)

'<|im_start|>system\nSYSTEM<|im_end|>\n<|im_start|>user\nQUESTION<|im_end|>\n<|im_start|>assistant\n'

In [16]:
tokenizer.apply_chat_template(messages, tokenize=False)

'<|im_start|>system\nSYSTEM<|im_end|>\n<|im_start|>user\nQUESTION<|im_end|>\n<|im_start|>assistant\nANSWER<|im_end|>\n'

In [19]:
response_template = "\n<|im_start|>assistant\n"
response_template_ids = tokenizer.encode(response_template)

In [18]:
print(states)

[151644, 8948, 198, 46487, 151645, 198, 151644, 872, 198, 52428, 151645, 198, 151644, 77091, 198, 11692, 39351, 151645, 198]


In [20]:
import torch

matches = (
    torch.tensor(states).unfold(0, len(response_template_ids), 1)
    .eq(torch.tensor(response_template_ids))
    .all(dim=1)
)
match_index = torch.nonzero(matches, as_tuple=False).flatten().tolist()[0]
actions =states[match_index+len(response_template_ids):]
action_mask = [0] * (match_index+len(response_template_ids)) + [1] * len(actions)
print(actions)
print(action_mask)

[11692, 39351, 151645, 198]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1]


In [21]:
print(tokenizer.decode([s*a for s,a in zip(states,action_mask) ]))

!!!!!!!!!!!!!!!ANSWER<|im_end|>



## MULTI-TURN

In [25]:
from trainer.datasets import base

SYSTEM_PROMPT = "Let's think step by step"
train_on_what = ["assistant"]

def determine_to_train(role: str) -> bool:
    return role in train_on_what    

messages = [
    base.Message(
            role="system",
            content=SYSTEM_PROMPT,
            train= determine_to_train("system"),
        ),
    base.Message(
        role="user",
        content="How are you?",
        train=determine_to_train("user"),
    ),
    base.Message(
        role="assistant",
        content="ANSWER-1",
        train=determine_to_train("assistant"),
    ),
    base.Message(
        role="user",
        content="Good evening",
        train=determine_to_train("user"),
    ),
    base.Message(
        role="assistant",
        content="ANSWER-2",
        train=determine_to_train("assistant"),
    ),
]

messages_dict = [{"role": m.role, "content": m.content} for m in messages]
messages_dict

[{'role': 'system', 'content': "Let's think step by step"},
 {'role': 'user', 'content': 'How are you?'},
 {'role': 'assistant', 'content': 'ANSWER-1'},
 {'role': 'user', 'content': 'Good evening'},
 {'role': 'assistant', 'content': 'ANSWER-2'}]

In [26]:
print(tokenizer.apply_chat_template(messages_dict,tokenize=False))

<|im_start|>system
Let's think step by step<|im_end|>
<|im_start|>user
How are you?<|im_end|>
<|im_start|>assistant
ANSWER-1<|im_end|>
<|im_start|>user
Good evening<|im_end|>
<|im_start|>assistant
ANSWER-2<|im_end|>



In [27]:
states = tokenizer.apply_chat_template(messages_dict,tokenize=True)

matches = (
    torch.tensor(states).unfold(0, len(response_template_ids), 1)
    .eq(torch.tensor(response_template_ids))
    .all(dim=1)
)
# We are indexing by -1 or the last assistant
match_index = torch.nonzero(matches, as_tuple=False).flatten().tolist()[-1]
actions = states[match_index+len(response_template_ids):]
action_mask = [0] * (match_index+len(response_template_ids)) + [1] * len(actions)
print(actions)
print(action_mask)

[11692, 39351, 12, 17, 151645, 198]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1]


In [28]:
tokenizer.decode(actions)

'ANSWER-2<|im_end|>\n'

In [29]:
multi_step_messages = [
    base.Message(
            role="system",
            content=SYSTEM_PROMPT,
            train= determine_to_train("system"),
        ),
    base.Message(
        role="user",
        content="How are you?",
        train=determine_to_train("user"),
    ),
    base.Message(
        role="assistant",
        content="ANSWER-1",
        train=determine_to_train("assistant"),
    ),
    base.Message(
        role="assistant",
        content="ANSWER-2",
        train=determine_to_train("assistant"),
    ),
]

In [30]:
multi_step_search_r1_messages = [
    base.Message(
            role="system",
            content=SYSTEM_PROMPT,
            train= determine_to_train("system"),
        ),
    base.Message(
        role="user",
        content="How are you?",
        train=determine_to_train("user"),
    ),
    base.Message(
        role="assistant",
        content="ANSWER-1",
        train=determine_to_train("assistant"),
    ),
    base.Message(
        role="tool",
        content="Retrieved documents",
        train=determine_to_train("tool"),
    ),
    base.Message(
        role="assistant",
        content="ANSWER-2",
        train=determine_to_train("assistant"),
    ),
]

In [36]:
from typing import List, Dict

def _tokenize_messages(
        messages: List[base.Message], rm: bool = False
    ) -> List[Dict[str, torch.Tensor]]:
        prev_text: str = ""
        states: List[int] = []
        actions: List[int] = []
        action_mask: List[bool] = []
        tensor_dicts: List[Dict[str, torch.Tensor]] = []

        def to_hf_messages(msgs: List[base.Message]) -> List[Dict[str, str]]:
            # HF chat templates expect {"role": ..., "content": ...}
            return [{"role": m.role, "content": m.content} for m in msgs]

        for turn in range(len(messages)):
            is_this_turn_train: bool = messages[turn].train
            is_next_turn_train: bool = (
                turn + 1 < len(messages) and messages[turn + 1].train
            )

            # Skip turns that are neither targets themselves nor part of the prompt
            # for an upcoming target turn. For example, if the current is system prompt
            if not is_this_turn_train and not is_next_turn_train:
                continue

            text: str = tokenizer.apply_chat_template(
                to_hf_messages(messages[: turn + 1]),
                add_generation_prompt=is_next_turn_train,
                tokenize=False,
            )

            if text.startswith(prev_text):
                delta_text_token_ids = tokenizer.encode(
                    text[len(prev_text) :], add_special_tokens=False
                )
                # Tokenize only the delta string to keep token sequence stable.
                delta_text_token_ids_len = len(delta_text_token_ids)
                states.extend(delta_text_token_ids)
                actions.extend(
                    delta_text_token_ids
                    if is_this_turn_train
                    else delta_text_token_ids_len * [0]
                )
                action_mask.extend(delta_text_token_ids_len * [is_this_turn_train])

            else:
                # Prefix broke (template rendering changed). We only allow a reset
                # right before an assistant/train turn (i.e., we are setting up a new prompt).
                assert (
                    is_next_turn_train
                ), "Template prefix broke at an unexpected point (not right before a train turn)."

                tensor_dicts.append(
                    base.get_tensor_dict(
                        states, actions, action_mask, 512, rm
                    )
                )

                states = tokenizer.encode(text, add_special_tokens=False)
                actions = [0] * len(states)
                action_mask = [False] * len(states)

            prev_text = text

        # Finalize last chunk
        tensor_dicts.append(
            base.get_tensor_dict(
                states, actions, action_mask, 512, rm
            )
        )
        return tensor_dicts

multi_step = _tokenize_messages(multi_step_search_r1_messages)
print(tokenizer.decode(multi_step[0]['states']))

<|im_start|>system
Let's think step by step<|im_end|>
<|im_start|>user
How are you?<|im_end|>
<|im_start|>assistant
ANSWER-1<|im_end|>
<|im_start|>user
<tool_response>
Retrieved documents
</tool_response><|im_end|>
<|im_start|>assistant
ANSWER-2<|im_end|>


In [37]:
print(tokenizer.decode(multi_step[0]['actions']))

!!!!!!!!!!!!!!!!!!!!!!ANSWER-1<|im_end|>
!!!!!!!!!!!!!!!!!!!!ANSWER-2<|im_end|>



In [38]:
print(tokenizer.decode(multi_step[0]['states'] * multi_step[0]['action_mask']))

!!!!!!!!!!!!!!!!!!!!!!
ANSWER-1<|im_end|>!!!!!!!!!!!!!!!!!!!!
ANSWER-2<|im_end|>
